In [1]:

# ============================================================
# 0. INSTALL + IMPORTS
# ============================================================

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics==8.3.*"
    ],
    check=True
)

from pathlib import Path
from collections import defaultdict, Counter
import shutil
import random
import hashlib
import json
import math

import numpy as np
import pandas as pd
import yaml

from ultralytics import YOLO

import zipfile
import csv
import random
from itertools import combinations
from IPython.display import display, FileLink
import cv2

import torch

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

COMPANY_ROOT = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/master-dataset"
)

PUBLIC_ROOT = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/ocr-dataset/Bangla LPDB - A/Bangla_License_Plate"
)

START_MODEL = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/sample-and-best-path/company_corrected_best/best.pt"
)

WORK_ROOT = Path(
    "/kaggle/working/final_rescue_training"
)

DATASET_ROOT = WORK_ROOT / "dataset"

FINAL_MODEL_ROOT = (
    WORK_ROOT / "final_model"
)

TRAIN_IMAGES = DATASET_ROOT / "images" / "train"
VAL_IMAGES = DATASET_ROOT / "images" / "val"
TEST_IMAGES = DATASET_ROOT / "images" / "test"

TRAIN_LABELS = DATASET_ROOT / "labels" / "train"
VAL_LABELS = DATASET_ROOT / "labels" / "val"
TEST_LABELS = DATASET_ROOT / "labels" / "test"

DATA_YAML = DATASET_ROOT / "data.yaml"

SEED = 42
NUM_CLASSES = 102

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

IMG_SIZE = 960
BATCH_SIZE = 4
EPOCHS = 35
PATIENCE = 8
DEVICE = [0,1]

IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

random.seed(SEED)
np.random.seed(SEED)

for p in [
    WORK_ROOT,
    TRAIN_IMAGES,
    VAL_IMAGES,
    TEST_IMAGES,
    TRAIN_LABELS,
    VAL_LABELS,
    TEST_LABELS,
    FINAL_MODEL_ROOT
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

if not COMPANY_ROOT.exists():
    raise FileNotFoundError(
        f"Company dataset not found:\n{COMPANY_ROOT}"
    )

if not PUBLIC_ROOT.exists():
    raise FileNotFoundError(
        f"LPDB-A dataset not found:\n{PUBLIC_ROOT}"
    )

if not START_MODEL.exists():
    raise FileNotFoundError(
        f"Known-good starting model not found:\n{START_MODEL}"
    )

print("Company data :", COMPANY_ROOT)
print("Public data  :", PUBLIC_ROOT)
print("Start model  :", START_MODEL)
print("GPU device   :", DEVICE)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Company data : /kaggle/input/datasets/tanvirmahtabtaneem/master-dataset
Public data  : /kaggle/input/datasets/tanvirmahtabtaneem/ocr-dataset/Bangla LPDB - A/Bangla_License_Plate
Start model  : /kaggle/input/datasets/tanvirmahtabtaneem/sample-and-best-path/company_corrected_best/best.pt
GPU device   : [0, 1]


In [2]:

# ============================================================
# 1. LOAD THE OFFICIAL 102 CLASS LIST
# ============================================================

CLASS_FILE = PUBLIC_ROOT / "classes.txt"

class_names = [
    x.strip()
    for x in CLASS_FILE.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()
    if x.strip()
]

if len(class_names) != NUM_CLASSES:
    raise RuntimeError(
        f"LPDB-A classes.txt contains {len(class_names)} classes, "
        f"not {NUM_CLASSES}."
    )

print(
    f"Official LPDB-A classes: {len(class_names)}"
)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def parse_label(text):
    rows = []

    for line in text.splitlines():

        line = line.strip()

        if not line:
            continue

        parts = line.split()

        if len(parts) != 5:
            return None

        try:
            cid = int(float(parts[0]))
            xc, yc, w, h = [
                float(v)
                for v in parts[1:]
            ]
        except Exception:
            return None

        if not (
            0 <= cid < NUM_CLASSES
        ):
            return None

        if not (
            0 <= xc <= 1
            and 0 <= yc <= 1
            and 0 < w <= 1
            and 0 < h <= 1
        ):
            return None

        rows.append(
            (
                cid,
                xc,
                yc,
                w,
                h
            )
        )

    return rows if rows else None


def label_canonical(text):
    rows = parse_label(text)

    if rows is None:
        return None

    return tuple(
        sorted(
            tuple(
                [row[0]]
                + [
                    round(x, 6)
                    for x in row[1:]
                ]
            )
            for row in rows
        )
    )


def md5_bytes(data):
    return hashlib.md5(
        data
    ).hexdigest()


def class_counter(records):
    counter = Counter()

    for r in records:

        rows = parse_label(
            r["label_text"]
        )

        if rows:

            for row in rows:
                counter[row[0]] += 1

    return counter


Official LPDB-A classes: 102


In [3]:

# ============================================================
# 2. READ COMPANY BATCHES
# ============================================================
#
# Actual structure:
#
# Batch 1/
#   images/images/train/*.jpg
#   fixed_annotation/obj_train_data/images/train/*.txt
#
# Future Batch 4, 5, 6 ... are detected automatically.
# ============================================================

company_records = []

batch_dirs = sorted(
    [
        p
        for p in COMPANY_ROOT.iterdir()
        if (
            p.is_dir()
            and p.name.lower().startswith("batch")
        )
    ],
    key=lambda p: int(
        "".join(
            x for x in p.name
            if x.isdigit()
        ) or 0
    )
)

print(
    "Batches found:",
    [
        p.name
        for p in batch_dirs
    ]
)

for batch_path in batch_dirs:

    image_dir = (
        batch_path
        / "images"
        / "images"
        / "train"
    )

    label_dir = (
        batch_path
        / "fixed_annotation"
        / "obj_train_data"
        / "images"
        / "train"
    )

    if not image_dir.exists():
        print(
            f"WARNING: {batch_path.name} "
            "image folder missing."
        )
        continue

    if not label_dir.exists():
        print(
            f"WARNING: {batch_path.name} "
            "label folder missing."
        )
        continue

    image_paths = [
        p
        for p in image_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTS
    ]

    missing = 0
    bad = 0
    good = 0

    for image_path in image_paths:

        label_path = (
            label_dir
            / f"{image_path.stem}.txt"
        )

        if not label_path.exists():

            missing += 1
            continue

        label_text = label_path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        rows = parse_label(
            label_text
        )

        if rows is None:

            bad += 1
            continue

        company_records.append({

            "source": "company",

            "batch":
                batch_path.name,

            "stem":
                image_path.stem,

            "image_path":
                image_path,

            "label_text":
                label_text,

            "md5":
                md5_bytes(
                    image_path.read_bytes()
                ),

            "canonical":
                label_canonical(
                    label_text
                )
        })

        good += 1

    print(
        f"{batch_path.name}: "
        f"images={len(image_paths)} | "
        f"usable={good} | "
        f"missing_label={missing} | "
        f"bad_label={bad}"
    )

print(
    "\nUsable company records:",
    len(company_records)
)


Batches found: ['Batch 1', 'Batch 2', 'Batch 3', 'Batch 4', 'Batch 5']
Batch 1: images=1296 | usable=1295 | missing_label=0 | bad_label=1
Batch 2: images=1309 | usable=1297 | missing_label=2 | bad_label=10
Batch 3: images=1275 | usable=1263 | missing_label=12 | bad_label=0
Batch 4: images=1811 | usable=1811 | missing_label=0 | bad_label=0
Batch 5: images=1269 | usable=1266 | missing_label=0 | bad_label=3

Usable company records: 6932


In [4]:

# ============================================================
# 3. CLEAN DUPLICATE COMPANY IMAGES
# ============================================================

by_md5 = defaultdict(list)

for r in company_records:
    by_md5[
        r["md5"]
    ].append(r)

cleaned_company = []
excluded_conflicts = []

for md5, group in by_md5.items():

    canonical_values = {
        r["canonical"]
        for r in group
    }

    if (
        len(canonical_values)
        > 1
    ):

        for r in group:

            excluded_conflicts.append({
                "batch":
                    r["batch"],
                "stem":
                    r["stem"],
                "reason":
                    "duplicate_image_conflicting_labels"
            })

        continue

    # Same image + same annotation:
    # keep only one copy.
    chosen = sorted(
        group,
        key=lambda x: (
            x["batch"],
            x["stem"]
        )
    )[0]

    cleaned_company.append(
        chosen
    )

print(
    "Company records before cleaning:",
    len(company_records)
)

print(
    "Company records after cleaning:",
    len(cleaned_company)
)

print(
    "Excluded conflicting duplicate copies:",
    len(excluded_conflicts)
)


# ------------------------------------------------------------
# COMPANY CLASS COVERAGE
# ------------------------------------------------------------

company_counter = class_counter(
    cleaned_company
)

company_present = set(
    company_counter.keys()
)

company_missing = sorted(
    set(range(NUM_CLASSES))
    - company_present
)

print(
    "\nCompany-only class coverage:"
)

print(
    "Classes present:",
    len(company_present),
    "/",
    NUM_CLASSES
)

print(
    "Classes missing:",
    len(company_missing)
)

print(
    "Missing class IDs:",
    company_missing
)


Company records before cleaning: 6932
Company records after cleaning: 6263
Excluded conflicting duplicate copies: 338

Company-only class coverage:
Classes present: 66 / 102
Classes missing: 36
Missing class IDs: [35, 37, 38, 41, 42, 43, 46, 47, 51, 53, 55, 56, 59, 61, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 78, 79, 80, 82, 88, 89, 90, 92, 93, 94, 95, 99]


In [5]:

# ============================================================
# 4. READ LPDB-A AND CREATE THE ORIGINAL 80/10/10 SPLIT
# ============================================================

public_images = sorted(
    [
        p
        for p in PUBLIC_ROOT.iterdir()
        if p.suffix.lower()
        in IMAGE_EXTS
    ]
)

public_pairs = []

for image_path in public_images:

    label_path = (
        image_path.with_suffix(".txt")
    )

    if not label_path.exists():
        continue

    label_text = label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    if parse_label(label_text) is None:
        continue

    public_pairs.append({

        "source": "public",

        "batch": "LPDB-A",

        "stem":
            image_path.stem,

        "image_path":
            image_path,

        "label_text":
            label_text,

        "md5":
            md5_bytes(
                image_path.read_bytes()
            ),

        "canonical":
            label_canonical(
                label_text
            )
    })


# Original seed-42 shuffle.
random.seed(SEED)

random.shuffle(
    public_pairs
)

total_public = len(
    public_pairs
)

public_train_end = int(
    total_public
    * TRAIN_RATIO
)

public_val_end = (
    public_train_end
    + int(
        total_public
        * VAL_RATIO
    )
)

public_train = public_pairs[
    :public_train_end
]

public_val = public_pairs[
    public_train_end:public_val_end
]

public_test = public_pairs[
    public_val_end:
]

print(
    "\nLPDB-A split:"
)

print(
    "Train:",
    len(public_train)
)

print(
    "Val:",
    len(public_val)
)

print(
    "Test:",
    len(public_test)
)



LPDB-A split:
Train: 2132
Val: 266
Test: 267


In [6]:

# ============================================================
# 5. SPLIT COMPANY DATA
# ============================================================

random.seed(SEED)

company_work = cleaned_company.copy()

random.shuffle(
    company_work
)

total_company = len(
    company_work
)

company_train_end = int(
    total_company
    * TRAIN_RATIO
)

company_val_end = (
    company_train_end
    + int(
        total_company
        * VAL_RATIO
    )
)

company_train = company_work[
    :company_train_end
]

company_val = company_work[
    company_train_end:company_val_end
]

company_test = company_work[
    company_val_end:
]

print(
    "\nCompany split:"
)

print(
    "Train:",
    len(company_train)
)

print(
    "Val:",
    len(company_val)
)

print(
    "Test:",
    len(company_test)
)


# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

final_train = (
    public_train
    + company_train
)

final_val = (
    public_val
    + company_val
)

final_test = (
    public_test
    + company_test
)

print(
    "\nCOMBINED:"
)

print(
    "Train:",
    len(final_train)
)

print(
    "Val  :",
    len(final_val)
)

print(
    "Test :",
    len(final_test)
)


# ------------------------------------------------------------
# CRITICAL 102-CLASS CHECK
# ------------------------------------------------------------

train_counter = class_counter(
    final_train
)

present_train = set(
    train_counter.keys()
)

missing_train = sorted(
    set(range(NUM_CLASSES))
    - present_train
)

print(
    "\nTraining class coverage:"
)

print(
    "Classes present:",
    len(present_train),
    "/",
    NUM_CLASSES
)

if missing_train:

    print(
        "MISSING FROM TRAINING:",
        missing_train
    )

    raise RuntimeError(
        "STOP. At least one of the 102 classes "
        "has zero training examples. "
        "Do not train until every class is represented."
    )

print(
    "SUCCESS: all 102 classes are represented "
    "in the final training set."
)



Company split:
Train: 5010
Val: 626
Test: 627

COMBINED:
Train: 7142
Val  : 892
Test : 894

Training class coverage:
Classes present: 102 / 102
SUCCESS: all 102 classes are represented in the final training set.


In [7]:

# ============================================================
# 6. WRITE THE FINAL MASTER DATASET
# ============================================================

def write_records(
    records,
    image_dir,
    label_dir,
    split_name
):

    manifest = []

    for index, r in enumerate(records):

        source_tag = (
            "public"
            if r["source"] == "public"
            else f"company_{r['batch'].replace(' ', '_')}"
        )

        new_stem = (
            f"{source_tag}_"
            f"{index:06d}_"
            f"{r['stem']}"
        )

        image_path = (
            image_dir
            / (
                new_stem
                + r["image_path"].suffix.lower()
            )
        )

        label_path = (
            label_dir
            / f"{new_stem}.txt"
        )

        shutil.copy2(
            r["image_path"],
            image_path
        )

        label_path.write_text(
            r["label_text"].strip()
            + "\n",
            encoding="utf-8"
        )

        manifest.append({
            "image":
                str(
                    image_path.relative_to(
                        DATASET_ROOT
                    )
                ),

            "label":
                str(
                    label_path.relative_to(
                        DATASET_ROOT
                    )
                ),

            "source":
                r["source"],

            "batch":
                r["batch"],

            "source_stem":
                r["stem"]
        })

    return manifest


manifest = []

manifest += write_records(
    final_train,
    TRAIN_IMAGES,
    TRAIN_LABELS,
    "train"
)

manifest += write_records(
    final_val,
    VAL_IMAGES,
    VAL_LABELS,
    "val"
)

manifest += write_records(
    final_test,
    TEST_IMAGES,
    TEST_LABELS,
    "test"
)


pd.DataFrame(
    manifest
).to_csv(
    DATASET_ROOT
    / "manifest.csv",
    index=False
)


data_config = {
    "path":
        str(DATASET_ROOT),

    "train":
        "images/train",

    "val":
        "images/val",

    "test":
        "images/test",

    "nc":
        NUM_CLASSES,

    "names":
        class_names
}

with open(
    DATA_YAML,
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False,
        allow_unicode=True
    )


print(
    "\nFinal master dataset written:"
)

print(
    DATASET_ROOT
)


# ------------------------------------------------------------
# FINAL CLASS DISTRIBUTION PRINT
# ------------------------------------------------------------

print(
    "\nFinal TRAIN class counts:"
)

for cid in range(
    NUM_CLASSES
):

    print(
        f"{cid:3d} | "
        f"{class_names[cid]:15s} | "
        f"{train_counter.get(cid, 0)}"
    )



Final master dataset written:
/kaggle/working/final_rescue_training/dataset

Final TRAIN class counts:
  0 | 0               | 4660
  1 | 1               | 9242
  2 | 2               | 4946
  3 | 3               | 4225
  4 | 4               | 3508
  5 | 5               | 4811
  6 | 6               | 4108
  7 | 7               | 2524
  8 | 8               | 3076
  9 | 9               | 1873
 10 | Metro           | 5457
 11 | A               | 48
 12 | Bha             | 55
 13 | Cha             | 1249
 14 | Chha            | 34
 15 | Da              | 114
 16 | DA              | 40
 17 | E               | 36
 18 | Ga              | 986
 19 | Gha             | 225
 20 | Ha              | 445
 21 | Ja              | 21
 22 | Jha             | 59
 23 | Ka              | 36
 24 | Kha             | 171
 25 | La              | 436
 26 | Ma              | 81
 27 | Na              | 197
 28 | Pa              | 17
 29 | Sa              | 13
 30 | Sha             | 46
 31 | Ta              | 1246

In [8]:

# ============================================================
# 7. FINAL TRAINING
# ============================================================
#
# IMPORTANT:
# We start from the LAST KNOWN GOOD model:
#
# company_corrected_best/best.pt
#
# NOT from the failed company-only final run.
# ============================================================

model = YOLO(
    str(START_MODEL)
)

results = model.train(

    data=str(DATA_YAML),

    imgsz=IMG_SIZE,

    batch=BATCH_SIZE,

    epochs=EPOCHS,

    patience=PATIENCE,

    device=DEVICE,

    optimizer="AdamW",

    lr0=0.0001,

    lrf=0.01,

    weight_decay=0.0005,

    degrees=0.0,

    translate=0.02,

    scale=0.10,

    shear=0.0,

    perspective=0.0,

    flipud=0.0,

    fliplr=0.0,

    hsv_h=0.01,

    hsv_s=0.20,

    hsv_v=0.20,

    mosaic=0.0,

    mixup=0.0,

    copy_paste=0.0,

    workers=2,

    cache=False,

    amp=True,

    project=str(
        FINAL_MODEL_ROOT
    ),

    name="bangla_final_102class",

    exist_ok=True,

    plots=True,

    verbose=True
)


BEST_MODEL = (
    FINAL_MODEL_ROOT
    / "bangla_final_102class"
    / "weights"
    / "best.pt"
)

if not BEST_MODEL.exists():

    raise FileNotFoundError(
        f"Final best.pt was not created:\n{BEST_MODEL}"
    )

print(
    "\nFINAL BEST MODEL:"
)

print(
    BEST_MODEL
)


New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/final_rescue_training/dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=35, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.2, hsv_v=0.2, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tra

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


       1/35      2.73G     0.6897     0.3399      1.077          9        960: 100% ━━━━━━━━━━━━ 1786/1786 4.6it/s 6:26
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 112/112 3.8it/s 29.2s
                   all        892       7855      0.926      0.873      0.942      0.741

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/35      2.73G     0.6474     0.3054      1.053         17        960: 93% ━━━━━━━━━━━─ 1661/1786 4.9it/s 5:36<25.3s

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


       2/35      2.73G      0.645     0.3043      1.051          9        960: 100% ━━━━━━━━━━━━ 1786/1786 5.0it/s 6:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 112/112 13.0it/s 8.6s
                   all        892       7855      0.906       0.88      0.941      0.741

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/35      2.73G     0.6325     0.2946      1.037         17        960: 11% ━─────────── 198/1786 5.0it/s 39.9s<5:15

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


... (17 outputs left out)

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


      12/35      2.77G      0.566     0.2512          1          9        960: 100% ━━━━━━━━━━━━ 1786/1786 5.0it/s 5:58
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 112/112 12.9it/s 8.7s
                   all        892       7855      0.927      0.893      0.949      0.747

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      13/35      2.77G     0.6038      0.262      1.018         18        960: 7% ╸─────────── 117/1786 4.9it/s 23.7s<5:44

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


      13/35      2.77G     0.5658     0.2526     0.9974          9        960: 100% ━━━━━━━━━━━━ 1786/1786 5.0it/s 5:58
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 112/112 12.9it/s 8.7s
                   all        892       7855      0.934      0.882      0.949      0.746
EarlyStopping: Training stopped early as no improvement observed in last 8 epochs. Best results observed at epoch 5, best model saved as best.pt.
To update EarlyStopping(patience=8) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

13 epochs completed in 1.352 hours.
Optimizer stripped from /kaggle/working/final_rescue_training/final_model/bangla_final_102class/weights/last.pt, 40.7MB
Optimizer stripped from /kaggle/working/final_rescue_training/final_model/bangla_final_102class/weights/best.pt, 40.7MB

Validating /kaggle/working/final_rescue_training/final_model/bangla_final_102class/weights/best.pt...
YOLO11

In [9]:

# ============================================================
# 8. DETECTION BENCHMARK
# ============================================================

final_model = YOLO(
    str(BEST_MODEL)
)

metrics = final_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    verbose=True
)

precision = float(
    metrics.box.mp
)

recall = float(
    metrics.box.mr
)

map50 = float(
    metrics.box.map50
)

map5095 = float(
    metrics.box.map
)

f1 = (
    2
    * precision
    * recall
    / (precision + recall)
    if precision + recall > 0
    else 0.0
)

benchmark = pd.DataFrame([{
    "precision": precision * 100,
    "recall": recall * 100,
    "F1": f1 * 100,
    "mAP50": map50 * 100,
    "mAP50-95": map5095 * 100
}])

print(
    "\n" + "=" * 65
)

print(
    "FINAL COMBINED HELD-OUT DETECTION BENCHMARK"
)

print(
    "=" * 65
)

print(
    benchmark.round(2).to_string(
        index=False
    )
)

benchmark.to_csv(
    WORK_ROOT
    / "final_detection_benchmark.csv",
    index=False
)


# ------------------------------------------------------------
# SAVE RUN SUMMARY
# ------------------------------------------------------------

summary = {
    "company_images_raw":
        len(company_records),

    "company_images_clean":
        len(cleaned_company),

    "company_train":
        len(company_train),

    "company_val":
        len(company_val),

    "company_test":
        len(company_test),

    "public_train":
        len(public_train),

    "public_val":
        len(public_val),

    "public_test":
        len(public_test),

    "final_train":
        len(final_train),

    "final_val":
        len(final_val),

    "final_test":
        len(final_test),

    "training_classes":
        len(present_train),

    "company_only_classes":
        len(company_present),

    "company_only_missing_class_ids":
        company_missing,

    "precision":
        precision,

    "recall":
        recall,

    "F1":
        f1,

    "mAP50":
        map50,

    "mAP50_95":
        map5095,

    "best_model":
        str(BEST_MODEL)
}

with open(
    WORK_ROOT
    / "final_rescue_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "\nFINAL RESCUE TRAINING COMPLETE."
)

print(
    "Use this best.pt:",
    BEST_MODEL
)


Ultralytics 8.3.253 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 125 layers, 20,108,674 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 319.9±174.5 MB/s, size: 10.4 KB)
val: Scanning /kaggle/working/final_rescue_training/dataset/labels/test... 894 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 894/894 1.2Kit/s 0.8s
val: /kaggle/working/final_rescue_training/dataset/images/test/public_000037_320.jpg: 1 duplicate labels removed
val: /kaggle/working/final_rescue_training/dataset/images/test/public_000125_197.jpeg: 1 duplicate labels removed
val: /kaggle/working/final_rescue_training/dataset/images/test/public_000140_325.jpg: 1 duplicate labels removed
val: New cache created: /kaggle/working/final_rescue_training/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP5

In [10]:
# ------------------------------------------------------------
# 9A. CHECK AND PACKAGE best.pt + data.yaml
# ------------------------------------------------------------

if not BEST_MODEL.is_file():
    raise FileNotFoundError(
        f"best.pt was not found:\n{BEST_MODEL}\n"
        "Run the training cell successfully before running this cell."
    )

if not class_names or len(class_names) != NUM_CLASSES:
    raise RuntimeError(
        f"Expected {NUM_CLASSES} class names; got {len(class_names)}."
    )

DELIVERY_DIR = WORK_ROOT / "model_delivery"
DELIVERY_DIR.mkdir(parents=True, exist_ok=True)

DELIVERY_MODEL = DELIVERY_DIR / "best.pt"
DELIVERY_YAML = DELIVERY_DIR / "data.yaml"

shutil.copy2(BEST_MODEL, DELIVERY_MODEL)

# Portable training dataset config. The dataset itself is not included
# in this two-file model package. If retraining on another machine,
# place the dataset under ./images and ./labels or edit these paths.
portable_data_config = {
    "path": ".",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": int(NUM_CLASSES),
    "names": [str(name) for name in class_names]
}

with open(DELIVERY_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        portable_data_config,
        f,
        sort_keys=False,
        allow_unicode=True
    )

DELIVERY_ZIP = Path("/kaggle/working/bangla_plate_model_package.zip")

with zipfile.ZipFile(DELIVERY_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(DELIVERY_MODEL, arcname="best.pt")
    zf.write(DELIVERY_YAML, arcname="data.yaml")

# Also put a copy alongside training outputs for easy discovery in Kaggle Output.
shutil.copy2(
    DELIVERY_ZIP,
    WORK_ROOT / "bangla_plate_model_package.zip"
)

# Validate the archive.
with zipfile.ZipFile(DELIVERY_ZIP, "r") as zf:
    if zf.testzip() is not None:
        raise RuntimeError("The exported model ZIP failed its integrity check.")
    if set(zf.namelist()) != {"best.pt", "data.yaml"}:
        raise RuntimeError(f"Unexpected ZIP contents: {zf.namelist()}")

print("=" * 70)
print("MODEL PACKAGE CREATED")
print("=" * 70)
print("Trained checkpoint :", BEST_MODEL)
print("ZIP package         :", DELIVERY_ZIP)
print("ZIP contents        : best.pt, data.yaml")
print("Portable YAML       :", DELIVERY_YAML)
print("\nDownload from the notebook link below or find the ZIP in Kaggle's Output files.")
display(FileLink(str(DELIVERY_ZIP)))

MODEL PACKAGE CREATED
Trained checkpoint : /kaggle/working/final_rescue_training/final_model/bangla_final_102class/weights/best.pt
ZIP package         : /kaggle/working/bangla_plate_model_package.zip
ZIP contents        : best.pt, data.yaml
Portable YAML       : /kaggle/working/final_rescue_training/model_delivery/data.yaml

Download from the notebook link below or find the ZIP in Kaggle's Output files.


/kaggle/working/bangla_plate_model_package.zip